# Covariate balance

Standardized mean difference and variance ratio, before and after matching. The manuscript treats a continuous covariate as balanced when the absolute standardized difference is below 0.1 and the variance ratio is between 0.5 and 2. Categorical covariates are matched exactly and are omitted here. The second table is the shared area of the two propensity-score histograms.

### Standardized difference and variance ratio

In [1]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

import pandas as pd
from core.balance import balance_before_after, propensity_overlap
from core.pipeline import ensure_synthetic, load_config

cfg = load_config(root)
frames = ensure_synthetic(root, cfg, rebuild=False)
pools = {"creator": "creators", "consumer": "consumers", "video": "videos", "algorithm": "algorithm"}
parts = []
overlap_rows = []
for task, pool_name in pools.items():
    spec = cfg["tasks"][task]
    pool = frames[pool_name]
    focal = pool[pool["treated"] == 1]
    comparison = pool[pool["treated"] == 0]
    pairs = pd.read_csv(root / "data" / "results" / f"{task}_pairs.csv")
    numeric = list(spec["numeric"])
    table = balance_before_after(focal, comparison, pairs, numeric)
    table.insert(0, "task", task)
    table["smd_below_0_1"] = table["standardized_difference_after"].abs() < 0.1
    table["variance_ratio_in_0_5_2"] = table["variance_ratio_after"].between(0.5, 2.0)
    parts.append(table)
    overlap_rows.append({
        "task": task,
        "overlap_before": propensity_overlap(focal, comparison, numeric),
        "overlap_after": propensity_overlap(
            pairs[[f"{column}_focal" for column in numeric]].rename(columns=lambda name: name[: -len("_focal")]),
            pairs[[f"{column}_comparison" for column in numeric]].rename(columns=lambda name: name[: -len("_comparison")]),
            numeric,
        ),
    })
balance = pd.concat(parts, ignore_index=True)
balance.to_csv(root / "data" / "results" / "balance_all.csv", index=False)
pd.DataFrame(overlap_rows).to_csv(root / "data" / "results" / "overlap.csv", index=False)
balance


,task,covariate,mean_focal_before,mean_comparison_before,standardized_difference_before,variance_ratio_before,mean_focal_after,mean_comparison_after,standardized_difference_after,variance_ratio_after,smd_below_0_1,variance_ratio_in_0_5_2
0,creator,followers,942.667500,893.493361,0.057603,1.096562,942.667500,820.969834,0.153943,1.563416,False,True
1,creator,plays_14d,97.997251,99.726962,-0.036495,0.950559,97.997251,94.261575,0.087362,1.491850,True,True
2,creator,plays_60d,299.140150,301.390721,-0.016198,0.962078,299.140150,285.437389,0.109196,1.506804,False,True
3,creator,uploads_14d,3.124588,3.164630,-0.035258,0.991621,3.124588,3.072618,0.049964,1.460403,True,True
4,creator,uploads_60d,7.878604,7.785207,0.032853,0.940209,7.878604,7.566407,0.119314,1.336560,False,True
5,creator,watch_time_14d,193.873975,196.340184,-0.030216,0.957399,193.873975,189.065377,0.065976,1.586556,True,True
6,creator,watch_time_60d,582.602695,591.549757,-0.036351,1.041818,582.602695,572.399588,0.045925,1.675492,True,True
7,consumer,age,31.943388,32.037152,-0.012019,1.006629,31.905186,32.050861,-0.019377,1.171895,True,True
8,consumer,creator_followers,774.913916,776.607925,-0.002742,1.057235,775.094217,730.507639,0.076896,1.282505,True,True
9,consumer,creator_plays_14d,88.422992,88.125955,0.008368,0.899102,88.586933,85.877726,0.079442,1.045459,True,True


### Propensity-score overlap

In [2]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

import pandas as pd
pd.read_csv(root / "data" / "results" / "overlap.csv")


,task,overlap_before,overlap_after
0,creator,0.942500,0.885000
1,consumer,0.968333,0.879684
2,video,0.952500,0.881667
3,algorithm,0.975833,0.879167
